# Preprocessing

Cleaning up the data before modeling. PCA cols (V1-V28) don't need scaling, 
already done. Just need to handle Time and Amount, then split.

## Import and load data

In [29]:
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import pandas as pd

df = pd.read_csv("../data/raw/creditcard.csv")

## Feature engineering - Time

Time is just seconds since first transaction, not useful as-is. 
Converting to hour of day instead.

In [30]:
df["Hour"] = (df["Time"] // 3600) % 24
df = df.drop(columns=["Time"])

## Train/test split

Splitting before scaling to avoid leakage. Using stratify to keep the ratio/percentage of fraud

In [31]:
X = df.drop(columns=["Class"])
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


## Scaling Amount

Amount has huge outliers so using RobustScaler instead of StandardScaler.
Fit only on train, then apply to test.

In [32]:
scaler = RobustScaler()
X_train["Amount_Scaled"] = scaler.fit_transform(X_train[["Amount"]])
X_test["Amount_Scaled"] = scaler.transform(X_test[["Amount"]])

# drop previous amount
X_train = X_train.drop(columns=["Amount"])
X_test = X_test.drop(columns=["Amount"])

## Save processed data

In [33]:
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

## Save the Scaler

In [34]:
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/amount_scaler.pkl")

['../models/amount_scaler.pkl']